# NB5｜應用二：食用油鑑別與摻混定量

**食品分析｜拉曼光譜與 RamanSPy 入門系列（第 5 本，共 5 本）**

橄欖油摻入便宜的葵花油是全球最常見的食品詐欺之一。這一本做兩件食品分析師的核心工作：**分類（是什麼油）** 和 **定量（摻了多少）**。

---
### 這一本你會學到
- 用 PCA + LDA 分辨四種食用油
- 用 PLS 迴歸建立摻混比例的檢量線
- 學會用交叉驗證選模型、用 RMSECV / R² 評估
- 理解「模型看的是哪個峰」為什麼比準確率更重要

> 💡 **完全沒寫過程式也沒關係。** 你只要做三件事：
> 1. 用滑鼠點每一格左邊的 ▶ 播放鍵（或按 `Shift + Enter`）
> 2. 看下面跑出來的圖和數字
> 3. 遇到 `# 👉 換你做` 的地方，照提示改一個數字或一個字，再跑一次


In [ ]:
# ===== 第一次執行請先跑這一格（大約 1 分鐘）=====
# 在 Google Colab 上，套件不是永久安裝的，每次重開都要跑一次。
!pip install -q ramanspy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ramanspy as rp

# 讓圖上的中文正常顯示（Colab 用）
!wget -q -O TaipeiSans.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_ 2>/dev/null
import matplotlib
try:
    matplotlib.font_manager.fontManager.addfont("TaipeiSans.ttf")
    matplotlib.rc("font", family="Taipei Sans TC Beta")
except Exception:
    pass
matplotlib.rcParams["axes.unicode_minus"] = False

print("準備完成！")


In [ ]:
# ===== 資料載入設定 =====
# 這一行由老師部署時自動填入正確的 GitHub 網址，學生不用改。
DATA_BASE = "https://raw.githubusercontent.com/Tai-ShengYeh/Tai-ShengYeh.github.io/main/ramanspy-food-analysis/data/"

# 若你把 CSV 直接上傳到 Colab 左側「檔案」，把上面那行改成： DATA_BASE = ""
# 若你在自己電腦跑，且 data 資料夾就在旁邊，改成：       DATA_BASE = "data/"

def load_spectra(filename):
    """讀 CSV → 回傳 (樣品資訊表 meta, ramanspy 光譜物件 spectra)"""
    df = pd.read_csv(DATA_BASE + filename)
    meta_cols = [c for c in df.columns if not c.replace(".", "", 1).isdigit()]
    axis = np.array([float(c) for c in df.columns if c not in meta_cols])
    spectra = rp.SpectralContainer(df.drop(columns=meta_cols).values, axis)
    return df[meta_cols].reset_index(drop=True), spectra

print("load_spectra() 已定義，資料來源：", DATA_BASE or "（Colab 本機檔案）")


In [ ]:
# 本課程統一使用的標準前處理流程
pipeline = rp.preprocessing.Pipeline([
    rp.preprocessing.misc.Cropper(region=(450, 1800)),          # 裁切
    rp.preprocessing.despike.WhitakerHayes(),                    # 去宇宙射線
    rp.preprocessing.denoise.SavGol(window_length=9, polyorder=3),  # 平滑
    rp.preprocessing.baseline.IModPoly(),                        # 基線校正
    rp.preprocessing.normalise.MinMax(),                         # 歸一化
])


## Part A｜分類：這是哪一種油？

In [ ]:
meta, spectra = load_spectra("edible_oils.csv")
proc = pipeline.apply(spectra)
X, axis = proc.spectral_data, proc.spectral_axis
y = meta.oil_type.values

print(meta.oil_type.value_counts())

In [ ]:
labels = {"olive": "橄欖油", "sunflower": "葵花油", "soybean": "大豆油", "coconut": "椰子油"}

plt.figure(figsize=(10, 4))
for k, lab in labels.items():
    plt.plot(axis, X[meta.oil_type == k].mean(0), lw=1.1, label=lab)
for w, txt in [(1265, "=C-H"), (1441, "CH$_2$"), (1523, "類胡蘿蔔素"), (1656, "C=C"), (1745, "C=O")]:
    plt.axvline(w, ls=":", lw=.7, color="grey")
    plt.text(w, 1.02, txt, fontsize=8, rotation=90, va="bottom")
plt.legend(); plt.xlabel("拉曼位移 (cm$^{-1}$)"); plt.title("四種食用油的平均光譜")
plt.show()

### 先用「化學家的方法」：算一個比值

不用機器學習，光是 **1265 / 1441 比值**（不飽和度指標）就能分開大部分油品。
**做分析永遠先試最簡單的方法。**

In [ ]:
i1265 = np.argmin(abs(axis - 1265))
i1441 = np.argmin(abs(axis - 1441))
ratio = X[:, i1265] / X[:, i1441]

for k, lab in labels.items():
    v = ratio[(meta.oil_type == k).values]
    print(f"{lab}：1265/1441 = {v.mean():.3f} ± {v.std():.3f}")

plt.figure(figsize=(7, 3.5))
plt.boxplot([ratio[(meta.oil_type == k).values] for k in labels])
plt.xticks(range(1, 5), list(labels.values()))
plt.ylabel("1265 / 1441 比值（不飽和度指標）")
plt.show()

### 再用機器學習：PCA + LDA

- **PCA** 先把 676 個波數壓成 6 個主成分（避免變數比樣品還多而過度配適）
- **LDA** 再找出最能分開四類的方向
- **交叉驗證**：把資料切 5 份，輪流拿 1 份當考卷、4 份當課本 —— 這才是誠實的成績

In [ ]:
from sklearn.decomposition import PCA as skPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix

model = make_pipeline(skPCA(n_components=6), LinearDiscriminantAnalysis())

scores = cross_val_score(model, X, y, cv=5)
print("5 折交叉驗證正確率：", scores.round(3))
print("平均 = %.1f%%" % (scores.mean() * 100))

pred = cross_val_predict(model, X, y, cv=5)
cm = pd.DataFrame(confusion_matrix(y, pred), index=sorted(set(y)), columns=sorted(set(y)))
print("\n混淆矩陣（列 = 真實，欄 = 預測）"); print(cm)

### 👉 換你做

把 `N_PC` 改成 2、3、10、20，看正確率怎麼變。

**思考**：主成分不是越多越好，為什麼？（提示：只有 60 個樣品）

In [ ]:
N_PC = 6      # 👉 換你做

m = make_pipeline(skPCA(n_components=N_PC), LinearDiscriminantAnalysis())
print(f"n_components={N_PC} → 交叉驗證正確率 = {cross_val_score(m, X, y, cv=5).mean()*100:.1f}%")

## Part B｜定量：橄欖油裡摻了多少葵花油？

50 個樣品，摻入比例 0–50%。這是典型的**多變量校正**問題，標準工具是 **PLS 迴歸**。

**PLS 一句話**：同時壓縮光譜和濃度，找出「跟濃度最相關」的光譜變化方向。

In [ ]:
ameta, aspectra = load_spectra("olive_adulteration.csv")
aproc = pipeline.apply(aspectra)
Xa, ya = aproc.spectral_data, ameta.sunflower_pct.values

print("樣品數：", len(ya), "｜ 摻入比例範圍：", ya.min(), "-", ya.max(), "%")

### 步驟 1：決定要用幾個潛在變數（LV）

- LV 太少 → **配適不足**，模型抓不到訊息
- LV 太多 → **過度配適**，模型把雜訊也背起來，對新樣品失準

做法：畫 **RMSECV vs LV** 曲線，取最低點（或最低點附近最小的 LV）。

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score

rmsecv = []
for n in range(1, 11):
    pred = cross_val_predict(PLSRegression(n_components=n), Xa, ya, cv=5).ravel()
    rmsecv.append(mean_squared_error(ya, pred) ** 0.5)

best_lv = int(np.argmin(rmsecv)) + 1

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, 11), rmsecv, "o-")
plt.plot(best_lv, rmsecv[best_lv-1], "o", ms=13, mfc="none", mec="crimson", mew=2)
plt.xlabel("潛在變數個數 (LV)"); plt.ylabel("RMSECV (%)")
plt.title(f"最佳 LV = {best_lv}")
plt.show()

for n, r in enumerate(rmsecv, 1):
    print(f"LV={n:2d}  RMSECV = {r:.2f} %")

### 步驟 2：建立模型並評估

In [ ]:
pls = PLSRegression(n_components=best_lv)
pred = cross_val_predict(pls, Xa, ya, cv=5).ravel()

rmse = mean_squared_error(ya, pred) ** 0.5
r2 = r2_score(ya, pred)

plt.figure(figsize=(5, 5))
plt.scatter(ya, pred, s=35)
plt.plot([0, 50], [0, 50], "--", color="grey")
plt.xlabel("實際摻入葵花油 (%)"); plt.ylabel("模型預測 (%)")
plt.title(f"RMSECV = {rmse:.2f} % ，R² = {r2:.3f}")
plt.show()

print(f"平均而言，模型的預測誤差約 ± {rmse:.1f} 個百分點。")
print(f"→ 也就是說，摻入 5% 以下的樣品，這個方法很可能抓不出來。")

### 步驟 3：檢查模型在看哪裡（絕對不能跳過）

In [ ]:
pls.fit(Xa, ya)
coef = pls.coef_.ravel()

plt.figure(figsize=(10, 3.5))
plt.plot(aproc.spectral_axis, coef, lw=1)
plt.axhline(0, color="grey", lw=.6)
for w in [1265, 1441, 1523, 1656]:
    plt.axvline(w, ls=":", lw=.8, color="crimson")
    plt.text(w, coef.max()*0.85, str(w), fontsize=8, rotation=90, color="crimson")
plt.xlabel("拉曼位移 (cm$^{-1}$)"); plt.ylabel("PLS 迴歸係數")
plt.title("模型倚重哪些波數？")
plt.show()

**判讀**：係數的大值應該落在 **1265（不飽和度）、1523（類胡蘿蔔素）、1656（C=C）** 附近 —— 這些正是橄欖油與葵花油真正的化學差異。

如果模型的大係數落在光譜的空白區或邊緣，那就是它抓到了雜訊或某種假相關，**準確率再高也不能信**。

## Part C｜綜合實作：判讀 6 個未知樣品

In [ ]:
umeta, uspectra = load_spectra("unknown_samples.csv")
uproc = pipeline.apply(uspectra)
Xu = uproc.spectral_data
uaxis = uproc.spectral_axis

key = {"478 澱粉": 478, "676 三聚氰胺": 676, "1003 蛋白質": 1003,
       "1085 醣類": 1085, "1441 油脂": 1441, "1523 類胡蘿蔔素": 1523, "1745 酯": 1745}
tab = pd.DataFrame({k: Xu[:, np.argmin(abs(uaxis - w))].round(3) for k, w in key.items()})
tab.insert(0, "樣品", umeta.sample_id)
tab

In [ ]:
# 👉 換你做：先自己從上表推理，再跑這一格看模型怎麼說
model.fit(X, y)                      # 用 Part A 的油品模型
油品判定 = model.predict(Xu)

for sid, p, oil_signal in zip(umeta.sample_id, 油品判定, Xu[:, np.argmin(abs(uaxis - 1745))]):
    note = labels.get(p, p) if oil_signal > 0.15 else "→ 1745 訊號太弱，恐怕不是油品，勿用此模型"
    print(f"{sid}：{note}")

> ⚠️ **上面示範了機器學習最危險的陷阱**：模型只會在它學過的四類裡挑一個，**它不會說「我不知道」**。
> 奶粉樣品丟進油品分類器，一樣會得到一個看起來很肯定的答案。
> 所以實務上一定要先做**樣品是否落在模型適用範圍內**的檢查（例如殘差、馬氏距離，或像這裡用 1745 cm⁻¹ 判斷是不是油）。

### 🧪 自我檢核

1. RMSECV = 3.3%，代表這個方法能不能可靠偵測 2% 的摻混？
2. 為什麼要用交叉驗證，而不是直接看模型對訓練資料的預測有多準？
3. PLS 的 LV 從 4 增加到 10，訓練誤差一定變小，但 RMSECV 卻上升 —— 這叫什麼現象？
4. 一個分類模型交叉驗證正確率 99%，但迴歸係數的大值落在光譜邊緣的空白區。你會怎麼做？
5. 把一個奶粉樣品丟進油品分類器，會發生什麼事？實務上怎麼防？

<details><summary>▶ 點開看參考答案</summary>

1. 不能。誤差 ±3.3 個百分點，2% 的摻混完全落在誤差範圍內。一般以 RMSECV 的 3 倍左右估計實用偵測下限，大約 10%。
2. 因為模型對自己看過的資料一定準（甚至可以 100%），那個數字不能代表它對新樣品的表現。交叉驗證模擬「遇到沒看過的樣品」。
3. 過度配適（overfitting）。模型開始把雜訊當成訊息記下來。
4. 不能採用。要回頭檢查是不是有系統性偏差（例如兩類樣品在不同天測、用不同批容器），這種假相關在真實實驗室非常常見。
5. 模型會硬把它歸到四類油之一，而且可能給出很高的信心值。要防範就要加上適用範圍（applicability domain）檢查：光譜殘差過大、或馬氏距離超出訓練集範圍時，回報「超出模型適用範圍」而不是給答案。

</details>

---
## 🎓 課程結束

你已經完成了一個完整的食品拉曼分析流程：

```
讀資料 → 前處理 → 峰位判讀 → 定性篩檢 → 定量校正 → 檢查模型合理性
```

**請到課程網站完成線上測驗，確認自己真的掌握了。**


---
### 📚 這一本用到的資料與文獻

**資料**：`data/` 內的光譜為**依文獻峰位建立的模擬資料**（`make_data.py`，亂數種子 20260801），刻意加入螢光背景、宇宙射線與雜訊。可用於教學演練，**不可引用為實驗證據**。

**主要文獻**

- Georgiev, D. et al. *RamanSPy: An Open-Source Python Package for Integrative Raman Spectroscopy Data Analysis*. **Anal. Chem.** 2024, 96(21), 8492–8500. doi:10.1021/acs.analchem.4c00383
- Gill, D.; Kilponen, R. G.; Rimai, L. *Resonance Raman Scattering … in Intact Plant Tissues*. **Nature** 1970, 227, 743–744. doi:10.1038/227743a0
- Lu, L. et al. *Resonance Raman scattering of β-carotene … second singlet state*. **J. Photochem. Photobiol. B** 2018, 179, 18–22. doi:10.1016/j.jphotobiol.2017.12.022
- Withnall, R. et al. *Raman spectra of carotenoids in natural products*. **Spectrochim. Acta A** 2003, 59(10), 2207–2212. doi:10.1016/S1386-1425(03)00064-7
- de Oliveira, V. E. et al. *Carotenes and carotenoids in natural biological samples*. **J. Raman Spectrosc.** 2010, 41(6), 642–650. doi:10.1002/jrs.2493
- Portarena, S. et al. *Cultivar discrimination, fatty acid profile and carotenoid characterization of monovarietal olive oils by Raman spectroscopy at a single glance*. **Food Control** 2019, 96, 137–145. doi:10.1016/j.foodcont.2018.09.011
- Chen, Y. et al. *Quantitative analysis of β-carotene and unsaturated fatty acids in blended olive oil via Raman spectroscopy combined with model prediction*. **Food Chemistry** 2025, 470, 142621. doi:10.1016/j.foodchem.2024.142621
- Schmidt, W. et al. *Continuous Temperature-Dependent Raman Spectroscopy of Melamine and Structural Analog Detection in Milk Powder*. **Appl. Spectrosc.** 2015, 69(3), 398–406. doi:10.1366/14-07600
- Zhang, X. et al. *Detection of melamine in liquid milk using SERS*. **J. Raman Spectrosc.** 2010, 41(12), 1655–1660. doi:10.1002/jrs.2629
- Kim, A. et al. *Melamine Sensing in Milk Products by Using SERS*. **Anal. Chem.** 2012, 84(21), 9303–9309. doi:10.1021/ac302025q
- FAO/WHO Codex Alimentarius. *General Standard for Contaminants and Toxins in Food and Feed*, **CXS 193-1995**.

完整清單見課程網站的「數據來源」與「參考文獻」兩節。